# OpenAI Codex Plugin — Research Notebook

This notebook integrates the **OpenAI Codex / GPT-4o** code-generation API into the research workflow.
It provides a lightweight plugin layer you can call from any other notebook in this repo.

**Features**
- Code generation & completion
- Code explanation / docstring generation
- Code review & refactoring suggestions
- Streaming responses

**Requirements**: An OpenAI API key stored in Colab Secrets as `OPENAI_API_KEY`.

In [ ]:
# Install dependencies
!pip install openai -q

In [ ]:
import os
from openai import OpenAI

# Load API key from Colab Secrets (recommended) or environment variable
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.environ.get('OPENAI_API_KEY')

if not api_key:
    raise ValueError('Set OPENAI_API_KEY in Colab Secrets or as an environment variable.')

client = OpenAI(api_key=api_key)
print('OpenAI client initialised.')

In [ ]:
# ── Codex Plugin Core ──────────────────────────────────────────────────────────

CODEX_MODEL = 'gpt-4o'  # swap for 'gpt-4o-mini' to reduce cost

def codex_generate(prompt: str, *, language: str = 'python', max_tokens: int = 1024) -> str:
    """Generate code from a natural-language prompt."""
    system = (
        f'You are an expert {language} programmer. '
        'Return only the requested code, no prose or markdown fences.'
    )
    response = client.chat.completions.create(
        model=CODEX_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()


def codex_explain(code: str) -> str:
    """Return a concise explanation of the given code snippet."""
    response = client.chat.completions.create(
        model=CODEX_MODEL,
        messages=[
            {'role': 'system', 'content': 'Explain the following code clearly and concisely.'},
            {'role': 'user',   'content': code},
        ],
        max_tokens=512,
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()


def codex_review(code: str) -> str:
    """Return code-review feedback with suggested improvements."""
    response = client.chat.completions.create(
        model=CODEX_MODEL,
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are a senior code reviewer. '
                    'Identify bugs, style issues, and performance improvements. '
                    'Be concise and actionable.'
                ),
            },
            {'role': 'user', 'content': code},
        ],
        max_tokens=768,
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()


def codex_complete(code_prefix: str, *, language: str = 'python', max_tokens: int = 512) -> str:
    """Complete a partial code snippet."""
    prompt = f'Complete the following {language} code:\n\n{code_prefix}'
    return codex_generate(prompt, language=language, max_tokens=max_tokens)


print('Codex plugin functions loaded: codex_generate, codex_explain, codex_review, codex_complete')

## Demo — Code Generation

In [ ]:
generated = codex_generate(
    'Write a Python function that downloads a Hugging Face model to a local cache directory '
    'and returns the local path. Use the huggingface_hub library.'
)
print(generated)

## Demo — Code Explanation

In [ ]:
sample_code = '''
def cosine_lr_schedule(step, warmup_steps, total_steps, min_lr, max_lr):
    import math
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))
'''

explanation = codex_explain(sample_code)
print(explanation)

## Demo — Code Review

In [ ]:
code_to_review = '''
import requests, json

def fetch_model_info(model_id):
    url = f"https://huggingface.co/api/models/{model_id}"
    r = requests.get(url)
    data = json.loads(r.text)
    return data["modelId"], data["downloads"]
'''

review = codex_review(code_to_review)
print(review)

## Using the Plugin from Other Notebooks

To use these helpers in another Colab notebook in this repo, add a setup cell:

```python
# In any other notebook:
!pip install openai -q

# Then copy/import the plugin functions above, or run this notebook's cells
# via %run if working locally.
```